# 04 Inference Submission

Thin Kaggle submission notebook using the shared baseline components.

In [ ]:
from pathlib import Path\nimport json\nimport sys\n\nROOT = Path.cwd().resolve()\nif not (ROOT / "src").exists():\n    for candidate in [ROOT, *ROOT.parents]:\n        if (candidate / "src").exists():\n            ROOT = candidate\n            break\n\nif not (ROOT / "src").exists() and Path("/kaggle/input").exists():\n    for nested in Path("/kaggle/input").rglob("src"):\n        if nested.is_dir():\n            ROOT = nested.parent\n            break\n\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nWORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT\nROOT, WORKDIR\n

In [ ]:
from src import load_config\nfrom src.data import load_eval_examples, resolve_dataset_path\nfrom src.eval import build_predictor, evaluate_examples\nfrom src.prompts import get_prompt_variant\nfrom src.solvers import ConservativeRouter\nfrom src.submission import write_submission_csv\n\nconfig = load_config(ROOT / "configs/experiments/baseline_eval.yaml")\ntest_path = resolve_dataset_path(\n    config["data"].get("test_path"),\n    fallback_filename="test.csv",\n    auto_discover=config["data"].get("auto_discover", False),\n    base_dir=ROOT,\n)\nprint("Using test file:", test_path)\n\nexamples = load_eval_examples(test_path)\nselected_variant = get_prompt_variant(config["selected_prompt_variant"])\nrouter = ConservativeRouter(\n    confidence_threshold=config["routing"]["confidence_threshold"],\n    enabled_families=tuple(config["routing"].get("enabled_families", [])) or None,\n)\npredictor = build_predictor(config)\n\nresult = evaluate_examples(\n    examples,\n    prompt_variants=[selected_variant],\n    predictor=predictor,\n    router=router if config["routing"]["enabled"] else None,\n    run_name="submission_generation",\n    output_dir=WORKDIR / config["kaggle"]["output_dir"],\n    failure_sample_size=config["reporting"].get("failure_sample_size", 5),\n)\nsubmission_path = write_submission_csv(\n    result.predictions,\n    WORKDIR / config["submission"]["output_csv"],\n)\nsubmission_path\n

In [ ]:
note_path = WORKDIR / config["kaggle"]["output_dir"] / "kaggle_run_note.json"\nnote_path.parent.mkdir(parents=True, exist_ok=True)\nnote_payload = {\n    "status": "submission_ready",\n    "selected_prompt_variant": selected_variant.name,\n    "submission_csv": str(submission_path),\n    "route_breakdown": result.metrics["route_breakdown"],\n    "next_steps": [\n        "Validate runtime and memory budget on the full Kaggle hardware path.",\n        "Inspect submission formatting before final submit.",\n        "Record the Kaggle checkpoint notes in reports/.",\n    ],\n}\nnote_path.write_text(json.dumps(note_payload, indent=2), encoding="utf-8")\nprint(note_path.read_text(encoding="utf-8"))\nprint((WORKDIR / config["submission"]["output_csv"]).read_text(encoding="utf-8"))\n